##### 1) EDA : size, info, shape 등
##### 2) 전체 테이블 데이터 자료 기준, 원하는 값(행,열)을 어떻게 찾아오고, 결측치는 어떻게 대체
> 전체 열 가운데, 국가별 필요한 값만 추출, 결측치를 제거
##### 3) 전체 테이블 내 특정 컬럼을 기준으로 그룹화하는 방법 & 그룹화된 상태에서 특정 조건에 해당되는 값만 조회
> 특정 국가는 한번만 사용되면 되기 때문에 그룹에 방법을 알아야 함
##### 4) 특정 컬럼 값 타입 변경
> 동적인 형태의 자료로 변경하려고 할 때, 정수의 형태를 띄고 있어야 하는 경우가 발생할 것임
##### 5) 서로 다른 데어터프레임간 병합
> 국가별 코로나 확진자수 추이 조회 : 2020-01-01 ~ 2023-12-31 : 하나의 데이터프레임 안에 각 개별 컬럼으로 존재
> 각 날짜의 데이터는 별도의 독립적인 csv
> 각 국가별 이름, 확진, 사망, 회복, 구각별 축약 키워드 : 하나의 데이터프레임 안에 모든 데이터들이 개별 컬럼으로 존재해야 함

In [1]:
import pandas as pd

df1 = pd.DataFrame({
    "id" : [1, 2, 3],
    "customer_id" : [1, 2, 3],
    "customer_name" : ["Robert", "Peter", "Dave"]
})

df1

,id,customer_id,customer_name
0,1,1,Robert
1,2,2,Peter
2,3,3,Dave


In [17]:
df2 = pd.DataFrame({
    "id" : [1, 2, 4],
    "order_id" : [100, 200, 300],
    "order_date" : ["2021-01-21", "2021-02-03", "2020-10-01"]
})

df2

,id,order_id,order_date
0,1,100,2021-01-21
1,2,200,2021-02-03
2,4,300,2020-10-01


In [ ]:
# 서로 다른 데이터 프레임 2개를 생성 > 한 개의 프레임으로 병합?!
# 병합의 방식 2가지
# 상·하
# 좌·우

In [3]:
pd.concat([df1,df2]) # 서로 다른 데이터프레임을 상·하 기준으로 합치는 것! -> 동일한 컬럼명을 가지고 있는 경우에만 정상적으로 값을 찾아오고, 그렇지 않은 경우 Nan으로 대체

,id,customer_id,customer_name,order_id,order_date
0,1,1.0,Robert,NaN,NaN
1,2,2.0,Peter,NaN,NaN
2,3,3.0,Dave,NaN,NaN
0,1,NaN,NaN,100.0,2021-01-21
1,2,NaN,NaN,200.0,2021-02-03
2,4,NaN,NaN,300.0,2020-10-01


In [ ]:
pd.concat([df1,df2], axis = 0)
# axis = 0 디폴트 값! -> 상·하 병합. axis = 1 좌·우병합

In [6]:
pd.concat([df1,df2], axis = 1) # 물리적 으로 교집합이 없더라도 일단 좌·우 병합

,id,customer_id,customer_name,id,order_id,order_date
0,1,1,Robert,1,100,2021-01-21
1,2,2,Peter,2,200,2021-02-03
2,3,3,Dave,4,300,2020-10-01


In [12]:
pd.merge(df1,df2) # 공통요소 = 컬럼을 기준으로 좌·우 병합 = INNER JOIN 
# on을 작성하지 않았어도 알아서 공통 걸럼명 찾음

,id,customer_id,customer_name,order_id,order_date
0,1,1,Robert,100,2021-01-21
1,2,2,Peter,200,2021-02-03


In [5]:
pd.merge(df1, df2, on = "id") # SQL, JOIN 특정 테이블 간 공통 컬럼 내 ON 전치사를 활용

,id,customer_id,customer_name,order_id,order_date
0,1,1,Robert,100,2021-01-21
1,2,2,Peter,200,2021-02-03


In [8]:
pd.merge(df1, df2, on = "id",how = "inner")

,id,customer_id,customer_name,order_id,order_date
0,1,1,Robert,100,2021-01-21
1,2,2,Peter,200,2021-02-03


In [9]:
pd.merge(df1, df2, on = "id",how = "outer") # left right 모두 살린 경우

,id,customer_id,customer_name,order_id,order_date
0,1,1.0,Robert,100.0,2021-01-21
1,2,2.0,Peter,200.0,2021-02-03
2,3,3.0,Dave,NaN,NaN
3,4,NaN,NaN,300.0,2020-10-01


In [10]:
pd.merge(df1, df2, on = "id",how = "left") # left

,id,customer_id,customer_name,order_id,order_date
0,1,1,Robert,100.0,2021-01-21
1,2,2,Peter,200.0,2021-02-03
2,3,3,Dave,NaN,NaN


In [11]:
pd.merge(df1, df2, on = "id",how = "right") #right

,id,customer_id,customer_name,order_id,order_date
0,1,1.0,Robert,100,2021-01-21
1,2,2.0,Peter,200,2021-02-03
2,4,NaN,NaN,300,2020-10-01


In [13]:
# 지금까지는 서로 다른 데이터프레임을 하나로 병합하는 조건, 특정 컬럼을 기준!!!
# 인덱스를 기준으로 병합
df1 = df1.set_index("id")
df1

,customer_id,customer_name
id,,
1,1,Robert
2,2,Peter
3,3,Dave


In [18]:
df2 = df2.set_index("id")
df2

,order_id,order_date
id,,
1,100,2021-01-21
2,200,2021-02-03
4,300,2020-10-01


In [19]:
pd.merge(df1, df2, left_index = True, right_index = True)

,customer_id,customer_name,order_id,order_date
id,,,,
1,1,Robert,100,2021-01-21
2,2,Peter,200,2021-02-03


In [20]:
pd.merge(df1, df2, how = "outer", left_index = True, right_index = True)
# 인덱스를 기준으로 병합을 하되, 레프트, 라이트 모두 아우터 생존

,customer_id,customer_name,order_id,order_date
id,,,,
1,1.0,Robert,100.0,2021-01-21
2,2.0,Peter,200.0,2021-02-03
3,3.0,Dave,NaN,NaN
4,NaN,NaN,300.0,2020-10-01


In [21]:
pd.merge(df1, df2, how = "left", left_index = True, right_index = True)
# 인덱스를 기준으로 병합을 하되, 레프트 아우터 생존

,customer_id,customer_name,order_id,order_date
id,,,,
1,1,Robert,100.0,2021-01-21
2,2,Peter,200.0,2021-02-03
3,3,Dave,NaN,NaN


In [22]:
pd.merge(df1, df2, how = "right", left_index = True, right_index = True)
# 인덱스를 기준으로 병합을 하되, 라이트 아우터 생존

,customer_id,customer_name,order_id,order_date
id,,,,
1,1.0,Robert,100,2021-01-21
2,2.0,Peter,200,2021-02-03
4,NaN,NaN,300,2020-10-01
